# DA6401 Assignment 1 – W&B Experiments

In [ ]:
import sys, os
sys.path.insert(0, 'src')   # adjust if running from a different directory

import numpy as np
import matplotlib.pyplot as plt
import wandb

from ann.neural_network import NeuralNetwork
from ann.optimizers import get_optimizer
from utils.data_loader import load_data
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

PROJECT = 'DA6401_Assignment_1_AM24M015'  
ENTITY  = None                     

wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Surya Pratap singh\_netrc.
wandb: Currently logged in as: spsinghiitian2020 (spsinghiitian2020-iitmaana) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 2.1  Data Exploration & Class Distribution

In [2]:
def log_sample_images(dataset='mnist'):
    (X_train, y_train, _, _, _, _, _, _, _) = load_data(dataset)

    class_names_mnist   = [str(i) for i in range(10)]
    class_names_fashion = ['T-shirt','Trouser','Pullover','Dress','Coat',
                           'Sandal','Shirt','Sneaker','Bag','Ankle boot']
    class_names = class_names_fashion if dataset == 'fashion_mnist' else class_names_mnist

    with wandb.init(project=PROJECT, entity=ENTITY,
                    name=f'data_exploration_{dataset}') as run:
        table = wandb.Table(columns=['class_id', 'class_name'] +
                                     [f'sample_{i+1}' for i in range(5)])
        for cls in range(10):
            idx = np.where(y_train == cls)[0][:5]
            images = [wandb.Image(X_train[i].reshape(28, 28), caption=class_names[cls])
                      for i in idx]
            table.add_data(cls, class_names[cls], *images)

        run.log({f'{dataset}_samples': table})
        print('Logged sample images to W&B')

log_sample_images('mnist')

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Logged sample images to W&B


## 2.2  Hyperparameter Sweep (≥100 runs)

In [3]:
# helper used by the sweep agent
def sweep_train(config=None):
    with wandb.init(config=config) as run:
        cfg = run.config

        (X_train, y_train, y_train_oh,
         X_val,   y_val,   y_val_oh,
         X_test,  y_test,  y_test_oh) = load_data(cfg.dataset)

        hidden_sizes = [cfg.hidden_size] * cfg.num_layers
        model = NeuralNetwork(784, hidden_sizes, 10,
                              activation=cfg.activation,
                              weight_init=cfg.weight_init,
                              loss=cfg.loss)
        opt = get_optimizer(cfg.optimizer, lr=cfg.learning_rate,
                            weight_decay=cfg.weight_decay)

        def get_batches(X, y, bs):
            idx = np.random.permutation(len(X))
            for s in range(0, len(X), bs):
                b = idx[s:s+bs]
                yield X[b], y[b]

        best_val = 0
        for epoch in range(cfg.epochs):
            for xb, yb in get_batches(X_train, y_train_oh, cfg.batch_size):
                logits = model.forward(xb)
                model.backward(logits, yb, weight_decay=cfg.weight_decay)
                opt.update(model.layers)

            val_acc = np.mean(model.predict_classes(X_val) == y_val)
            train_acc = np.mean(model.predict_classes(X_train) == y_train)
            train_loss = model.compute_loss(model.forward(X_train[:512]), y_train_oh[:512])
            best_val = max(best_val, val_acc)

            wandb.log({'epoch': epoch+1, 'val_acc': val_acc,
                       'train_acc': train_acc, 'train_loss': train_loss})

        wandb.log({'best_val_acc': best_val,
                   'test_acc': np.mean(model.predict_classes(X_test) == y_test)})


sweep_config = {
    'method': 'bayes',   # bayes search reaches 100 runs faster than grid
    'metric': {'name': 'best_val_acc', 'goal': 'maximize'},
    'parameters': {
        'dataset':       {'value': 'mnist'},
        'epochs':        {'value': 10},
        'batch_size':    {'values': [32, 64, 128]},
        'loss':          {'values': ['cross_entropy', 'mean_squared_error']},
        'optimizer':     {'values': ['sgd', 'momentum', 'nag', 'rmsprop', 'adam', 'nadam']},
        'learning_rate': {'distribution': 'log_uniform_values', 'min': 1e-4, 'max': 1e-1},
        'weight_decay':  {'values': [0.0, 1e-4, 1e-3]},
        'num_layers':    {'values': [1, 2, 3, 4 ,5 ,6]},
        'hidden_size':   {'values': [32, 64, 128]},
        'activation':    {'values': ['sigmoid', 'tanh', 'relu']},
        'weight_init':   {'values': ['random', 'xavier']},
    }
}

sweep_id = wandb.sweep(sweep_config, project=PROJECT, entity=ENTITY)
print(f'Sweep ID: {sweep_id}')
wandb.agent(sweep_id, sweep_train, count=50)

Create sweep with ID: vakntww0
Sweep URL: https://wandb.ai/spsinghiitian2020-iitmaana/da6401_mlp_assignment/sweeps/vakntww0
Sweep ID: vakntww0


wandb: Agent Starting Run: yycyhtod with config:
wandb: 	activation: relu
wandb: 	batch_size: 32
wandb: 	dataset: mnist
wandb: 	epochs: 10
wandb: 	hidden_size: 128
wandb: 	learning_rate: 0.009505948833720943
wandb: 	loss: cross_entropy
wandb: 	num_layers: 5
wandb: 	optimizer: momentum
wandb: 	weight_decay: 0.0001
wandb: 	weight_init: random
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Surya Pratap singh\_netrc.


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


best_val_acc,▁
epoch,▁▂▃▃▄▅▆▆▇█
test_acc,▁
train_acc,▁▁▁▁▁▁▁▁▁▁
train_loss,█▅▄▂▂▁▅▅▃▃
val_acc,▁▁▁▁▁▁▁▁▁▁
best_val_acc,0.11183
epoch,10
test_acc,0.1135
train_acc,0.11243
train_loss,2.30092


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: w161lfnd with config:
wandb: 	activation: tanh
wandb: 	batch_size: 128
wandb: 	dataset: mnist
wandb: 	epochs: 10
wandb: 	hidden_size: 32
wandb: 	learning_rate: 0.018261722858953583
wandb: 	loss: cross_entropy
wandb: 	num_layers: 6
wandb: 	optimizer: nag
wandb: 	weight_decay: 0.0001
wandb: 	weight_init: random
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Surya Pratap singh\_netrc.


...

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


best_val_acc,▁
epoch,▁▂▃▃▄▅▆▆▇█
test_acc,▁
train_acc,▁▄▆▆▆▆▅▇██
train_loss,█▄▂▄▂▃▇▃▃▁
val_acc,▁▅▆▇▇▆▆██▇
best_val_acc,0.95367
epoch,10
test_acc,0.952
train_acc,0.95667
train_loss,0.10334


wandb: Agent Starting Run: 1g6o11zv with config:
wandb: 	activation: relu
wandb: 	batch_size: 64
wandb: 	dataset: mnist
wandb: 	epochs: 10
wandb: 	hidden_size: 32
wandb: 	learning_rate: 0.016367222427782834
wandb: 	loss: cross_entropy
wandb: 	num_layers: 1
wandb: 	optimizer: adam
wandb: 	weight_decay: 0
wandb: 	weight_init: xavier
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Surya Pratap singh\_netrc.


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


best_val_acc,▁
epoch,▁▂▃▃▄▅▆▆▇█
test_acc,▁
train_acc,▄▁▅▇▇▆█▇█▇
train_loss,▄▆▄▂▃█▁▁▁▂
val_acc,▆▁▅█▆▅▅▅█▄
best_val_acc,0.955
epoch,10
test_acc,0.9459
train_acc,0.96041
train_loss,0.10734


## 2.3  Optimizer Showdown

In [4]:
OPTIMIZERS = ['sgd', 'momentum', 'nag', 'rmsprop', 'adam', 'nadam']

(X_train, y_train, y_train_oh,
 X_val,   y_val,   y_val_oh, _, _, _) = load_data('mnist')

def get_batches(X, y, bs):
    idx = np.random.permutation(len(X))
    for s in range(0, len(X), bs):
        b = idx[s:s+bs]
        yield X[b], y[b]

for opt_name in OPTIMIZERS:
    with wandb.init(project=PROJECT, entity=ENTITY,
                    name=f'optimizer_cmp_{opt_name}',
                    group='optimizer_showdown') as run:

        model = NeuralNetwork(784, [128, 128, 128], 10,
                              activation='relu', weight_init='xavier',
                              loss='cross_entropy')
        opt = get_optimizer(opt_name, lr=1e-3)

        for epoch in range(5):
            epoch_losses = []
            for xb, yb in get_batches(X_train, y_train_oh, 64):
                logits = model.forward(xb)
                loss   = model.compute_loss(logits, yb)
                model.backward(logits, yb)
                opt.update(model.layers)
                epoch_losses.append(loss)

            wandb.log({'epoch': epoch+1,
                       'train_loss': np.mean(epoch_losses),
                       'val_acc': np.mean(model.predict_classes(X_val) == y_val)})
        print(f'{opt_name} done')

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


sgd done


epoch,▁▃▅▆█
train_loss,█▇▅▂▁
val_acc,▁▅▆▇█
epoch,5
train_loss,0.82836
val_acc,0.81517


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


momentum done


...

adam done


epoch,▁▃▅▆█
train_loss,█▃▂▁▁
val_acc,▁▆███
epoch,5
train_loss,0.04408
val_acc,0.97617


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


nadam done


epoch,▁▃▅▆█
train_loss,█▃▂▁▁
val_acc,▁▅▅▇█
epoch,5
train_loss,0.04347
val_acc,0.97833


## 2.4  Vanishing Gradient Analysis (Sigmoid vs ReLU)

In [5]:
for act in ['sigmoid', 'relu']:
    for n_layers in [3, 5]:  # shallower and deeper
        with wandb.init(project=PROJECT, entity=ENTITY,
                        name=f'vanishing_grad_{act}_L{n_layers}',
                        group='vanishing_gradient') as run:

            model = NeuralNetwork(784, [64]*n_layers, 10,
                                  activation=act, weight_init='xavier',
                                  loss='cross_entropy')
            opt = get_optimizer('adam', lr=1e-3)

            for epoch in range(10):
                for xb, yb in get_batches(X_train, y_train_oh, 128):
                    logits = model.forward(xb)
                    model.backward(logits, yb)
                    opt.update(model.layers)

                grad_norm = np.linalg.norm(model.layers[0].grad_W)
                val_acc   = np.mean(model.predict_classes(X_val) == y_val)
                wandb.log({'epoch': epoch+1,
                           'grad_norm_layer0': grad_norm,
                           'val_acc': val_acc})

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▂▃▃▄▅▆▆▇█
grad_norm_layer0,▁▄▃▄▇▁█▃▂▆
val_acc,▁▄▆▆▇▇████
epoch,10
grad_norm_layer0,0.20143
val_acc,0.971


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▂▃▃▄▅▆▆▇█
grad_norm_layer0,▂▅▂▄█▁▄▆▄▆
val_acc,▁▆▇███████
epoch,10
grad_norm_layer0,0.36583
val_acc,0.95883


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▂▃▃▄▅▆▆▇█
grad_norm_layer0,▄▇▇▄█▅▇▆▆▁
val_acc,▁▄▆▇▇█▇▇██
epoch,10
grad_norm_layer0,0.2404
val_acc,0.972


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▂▃▃▄▅▆▆▇█
grad_norm_layer0,▅▃▃▄▄▃█▂▁▃
val_acc,▁▅▇▇▇▇▆█▇█
epoch,10
grad_norm_layer0,0.625
val_acc,0.97167


## 2.5  Dead Neuron Investigation (ReLU high-LR vs Tanh)

In [6]:
for act, lr in [('relu', 0.1), ('tanh', 0.1)]:
    with wandb.init(project=PROJECT, entity=ENTITY,
                    name=f'dead_neuron_{act}_lr{lr}',
                    group='dead_neurons') as run:

        model = NeuralNetwork(784, [128, 128], 10,
                              activation=act, weight_init='xavier',
                              loss='cross_entropy')
        opt = get_optimizer('sgd', lr=lr)

        for epoch in range(10):
            for xb, yb in get_batches(X_train, y_train_oh, 64):
                logits = model.forward(xb)
                model.backward(logits, yb)
                opt.update(model.layers)

            # fraction of neurons with zero activation across the whole val set
            _ = model.forward(X_val[:1000])
            dead = float(np.mean(model.layers[0].A == 0))
            val_acc = np.mean(model.predict_classes(X_val) == y_val)
            wandb.log({'epoch': epoch+1,
                       'dead_neuron_fraction': dead,
                       'val_acc': val_acc,
                       'grad_norm': np.linalg.norm(model.layers[0].grad_W)})

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


dead_neuron_fraction,▁▄▅▆▇██▇▇█
epoch,▁▂▃▃▄▅▆▆▇█
grad_norm,▆▃▃▂█▄▁▅▇▆
val_acc,▁▄▆▇▇█████
dead_neuron_fraction,0.49371
epoch,10
grad_norm,0.99159
val_acc,0.97467


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


dead_neuron_fraction,▁▁▁▁▁▁▁▁▁▁
epoch,▁▂▃▃▄▅▆▆▇█
grad_norm,█▅▇▆▂▄▆▁▅▄
val_acc,▁▄▅▇▇▇▇███
dead_neuron_fraction,0
epoch,10
grad_norm,0.61865
val_acc,0.9765


## 2.6  Loss Function Comparison (MSE vs Cross-Entropy)

In [7]:
for loss_fn in ['cross_entropy', 'mean_squared_error']:
    with wandb.init(project=PROJECT, entity=ENTITY,
                    name=f'loss_cmp_{loss_fn}',
                    group='loss_comparison') as run:

        model = NeuralNetwork(784, [128, 128], 10,
                              activation='relu', weight_init='xavier',
                              loss=loss_fn)
        opt = get_optimizer('adam', lr=1e-3)

        for epoch in range(15):
            epoch_losses = []
            for xb, yb in get_batches(X_train, y_train_oh, 64):
                logits = model.forward(xb)
                loss   = model.compute_loss(logits, yb)
                model.backward(logits, yb)
                opt.update(model.layers)
                epoch_losses.append(loss)

            wandb.log({'epoch': epoch+1,
                       'train_loss': np.mean(epoch_losses),
                       'val_acc': np.mean(model.predict_classes(X_val) == y_val)})

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_loss,█▄▃▂▂▂▂▁▁▁▁▁▁▁▁
val_acc,▁▆▆▅▇▇▇▇▇▆██▅▅█
epoch,15
train_loss,0.01003
val_acc,0.97817


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_loss,█▄▃▂▂▂▂▁▁▁▁▁▁▁▁
val_acc,▁▄▅▅▆█▇█▇▇█▇▇█▆
epoch,15
train_loss,0.00104
val_acc,0.97367


## 2.9  Weight Init & Symmetry (Zeros vs Xavier)

In [8]:
for w_init in ['zeros', 'xavier']:
    with wandb.init(project=PROJECT, entity=ENTITY,
                    name=f'weight_init_{w_init}',
                    group='weight_init_symmetry') as run:

        model = NeuralNetwork(784, [128, 128], 10,
                              activation='relu', weight_init=w_init,
                              loss='cross_entropy')
        opt = get_optimizer('sgd', lr=0.01)

        # log gradients of 5 specific neurons for first 50 iterations
        iteration = 0
        NEURON_IDS = [0, 10, 25, 50, 100]

        for xb, yb in get_batches(X_train, y_train_oh, 64):
            if iteration >= 50:
                break

            logits = model.forward(xb)
            model.backward(logits, yb)
            opt.update(model.layers)

            grad_log = {f'neuron_{n}_grad': float(np.linalg.norm(model.layers[0].grad_W[:, n]))
                        for n in NEURON_IDS}
            grad_log['iteration'] = iteration + 1
            wandb.log(grad_log)
            iteration += 1

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


iteration,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇██
neuron_0_grad,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
neuron_100_grad,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
neuron_10_grad,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
neuron_25_grad,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
neuron_50_grad,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
iteration,50
neuron_0_grad,0
neuron_100_grad,0
neuron_10_grad,0
neuron_25_grad,0


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


iteration,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇███
neuron_0_grad,▃▃▃▁▂▃▃▇▁▄▃▃▆▃▄▃▄█▅▇█▂▅▃▄▆▂▄▅▄▂▅▄▄▃▄▂▃▄▃
neuron_100_grad,▆▄▄▅▂▃█▄▃▂▄▃▄▅▃▇▆█▄▅▃▃█▃▅▅▆▅▇▆▃▃▃▂▅▄▁▃▃▂
neuron_10_grad,▃▅▃▂▅▄▅▃▅▂▂▂▄▆▄▃▁▂▂▃▁▃▃▃▃█▁▃▂▄▃▃▂▄▃▄▃▂▂▄
neuron_25_grad,▃█▅▂▃▁▅▂▃▂▅▃▃▅▂▃▇█▃▂▄▄▂▂▂▃▇▄▂▃▂▅▄▃▅▃▂▃▂▃
neuron_50_grad,▃▅▂▃▂▃▃▃▂▃▁▄▅▃▁▃▂▂▇▆▄▅▄▂▁█▃▂▃▃▆▆▃▃▂▂▅█▁▂
iteration,50
neuron_0_grad,0.00866
neuron_100_grad,0.08891
neuron_10_grad,0.12401
neuron_25_grad,0.12185


## 2.10  Fashion-MNIST Transfer Challenge (3 configs)

In [15]:
from tensorflow.keras.datasets import fashion_mnist
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()

import numpy as np
np.savez('fashion_mnist.npz',
         x_train=X_train_full,
         y_train=y_train_full,
         x_test=X_test,
         y_test=y_test)
print("Saved fashion_mnist.npz")

Saved fashion_mnist.npz


In [14]:
# Based on MNIST sweep findings, pick your 3 best configs here
fashion_configs = [
    {'optimizer': 'adam',  'activation': 'relu', 'num_layers': 3, 'hidden_size': 128,
     'learning_rate': 1e-3, 'batch_size': 64, 'weight_init': 'xavier'},
    {'optimizer': 'nadam', 'activation': 'relu', 'num_layers': 4, 'hidden_size': 128,
     'learning_rate': 5e-4, 'batch_size': 64, 'weight_init': 'xavier'},
    {'optimizer': 'adam',  'activation': 'tanh', 'num_layers': 3, 'hidden_size': 128,
     'learning_rate': 1e-3, 'batch_size': 128, 'weight_init': 'xavier'},
]

(X_tr, y_tr, y_tr_oh,
 X_v,  y_v,  y_v_oh,
 X_te, y_te, y_te_oh) = load_data('fashion_mnist')

for i, fc in enumerate(fashion_configs):
    with wandb.init(project=PROJECT, entity=ENTITY,
                    name=f'fashion_config_{i+1}',
                    group='fashion_transfer',
                    config={**fc, 'dataset': 'fashion_mnist', 'epochs': 15,
                            'loss': 'cross_entropy'}) as run:

        model = NeuralNetwork(784, [fc['hidden_size']]*fc['num_layers'], 10,
                              activation=fc['activation'],
                              weight_init=fc['weight_init'],
                              loss='cross_entropy')
        opt = get_optimizer(fc['optimizer'], lr=fc['learning_rate'])

        for epoch in range(15):
            for xb, yb in get_batches(X_tr, y_tr_oh, fc['batch_size']):
                logits = model.forward(xb)
                model.backward(logits, yb)
                opt.update(model.layers)

            va = np.mean(model.predict_classes(X_v) == y_v)
            wandb.log({'epoch': epoch+1, 'val_acc': va})

        test_acc = np.mean(model.predict_classes(X_te) == y_te)
        wandb.log({'test_acc': test_acc})
        print(f'Config {i+1} → test_acc = {test_acc:.4f}')

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Config 1 → test_acc = 0.8823


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_acc,▁
val_acc,▁▃▅▅▅▆▅▅█▆▇▇▆▇▇
epoch,15
test_acc,0.8823
val_acc,0.8945


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Config 2 → test_acc = 0.8879


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_acc,▁
val_acc,▁▃▄▄▅▆▅▇▇▆█▇█▆▇
epoch,15
test_acc,0.8879
val_acc,0.889


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Config 3 → test_acc = 0.8817


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_acc,▁
val_acc,▁▂▄▄▃▆▄▅▇▇█▇█▅▇
epoch,15
test_acc,0.8817
val_acc,0.88967
